# ML Assignment 02
### Dataset: House Price Prediction Dataset
### Total Marks: 100

---

## Exam Instructions:
1. প্রথমে নিচের cell এ নিজের **নাম** এবং কোর্সে registration করা **ইমেইল** দিবে
2. Question wise numbering করে Text cell রাখবে এবং এর নিচে Code cell থাকবে, চেষ্টা করবে একটি code cell এ একটি question উত্তর দেওয়ার
3. Google Colab এর মধ্যে কোডগুলো করবে
4. এবং সেই ফাইলটি **'Anyone with the link' & 'View' Access** দিয়ে ফাইলটির Shareable Link টি সাবমিট করবে

---

**Question Dataset Link:** https://www.kaggle.com/datasets/prokshitha/home-value-insights

## Student Information

In [1]:
# Fill in your information
name = "Tanzim Ahamed"           # Write your full name here
email = "tanzim.ahamed.bd@gmail.com"          # Write your registered email here

print(f"Name  : {name}")
print(f"Email : {email}")

Name  : Tanzim Ahamed
Email : tanzim.ahamed.bd@gmail.com


---
## Question 1 (10 Marks)

Load the House Price dataset and display:
- Dataset shape
- First 10 rows
- 5 random samples

In [2]:
# Question 1
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
df=pd.read_csv("/content/house_price_regression_dataset.csv")


df.shape
df.head(10)
df.sample(5)

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
931,2579,2,3,2010,2.700661,1,5,603812.588792
516,3433,3,2,2004,1.116903,2,5,739205.515933
148,1198,3,2,2012,3.866021,2,5,346152.847592
412,912,4,1,2008,1.055304,0,5,233504.513784
190,4561,2,1,1954,2.171687,2,2,907358.632996


---
## Question 2 (10 Marks)

Handle missing values and perform feature engineering:
- Impute missing numerical values using `SimpleImputer` with mean strategy
- Impute missing categorical values using most frequent strategy
- Drop columns with more than 50% missing values
- Perform train-test split with `test_size=0.2` and `random_state=42`

Display the shape of final train and test sets.

In [18]:
df.info()
df.dtypes

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 8 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   Square_Footage        1000 non-null   float64
 1   Num_Bedrooms          1000 non-null   float64
 2   Num_Bathrooms         1000 non-null   float64
 3   Year_Built            1000 non-null   float64
 4   Lot_Size              1000 non-null   float64
 5   Garage_Size           1000 non-null   float64
 6   Neighborhood_Quality  1000 non-null   float64
 7   House_Price           1000 non-null   float64
dtypes: float64(8)
memory usage: 62.6 KB


,0
Square_Footage,float64
Num_Bedrooms,float64
Num_Bathrooms,float64
Year_Built,float64
Lot_Size,float64
Garage_Size,float64
Neighborhood_Quality,float64
House_Price,float64


In [7]:
#Question 2
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split

In [8]:
# Drop columns with more than 50% missing values

df = df.loc[:, df.isnull().mean() < 0.5]
df.shape



(1000, 8)

In [19]:
# Numerical columns

num_cols = df.select_dtypes(include=['float64','int64']).columns

In [20]:
# Mean Imputer

num_imputer = SimpleImputer(strategy='mean')

df[num_cols] = num_imputer.fit_transform(df[num_cols])

In [21]:
# Categorical Imputer
# cat_imputer = SimpleImputer(strategy='most_frequent')
# df[cat_cols] = cat_imputer.fit_transform(df[cat_cols])

In [22]:
X = df.drop('House_Price', axis=1)
y = df['House_Price']





X_train, X_test, y_train, y_test = train_test_split(
                                                    X,
                                                    y,
                                                    test_size=0.2,
                                                    random_state=42
                                                     )

In [23]:
print(X_train.shape)
print(y_train.shape)


(800, 7)
(800,)


---
## Question 3 (20 Marks)

Implement **Simple Linear Regression** using **only NumPy** (no Scikit-Learn allowed):
- Compute slope (`m`) and intercept (`c`) using the Batch Gradient Descent
- Predict values for the test set
- Print the learned `m` and `c` values

Use `Square_Footage` as feature (X) and `House_Price` as target (y).

In [25]:
#Question 3

def make_prediction(X, w, b):
    m = X.shape[0]
    predictions = np.zeros(m)
    for i in range(m):
        predictions[i] = w * X[i] + b

    return predictions



X = X_train['Square_Footage'].values
y = y_train.values

In [27]:
w = 0
b = 0

learning_rate = 0.00000001
epochs = 1000
m = X.shape[0]

for i in range(epochs):
    predictions = make_prediction(X, w, b)
    dw = (-2/m) * np.sum(X * (y - predictions))
    db = (-2/m) * np.sum(y - predictions)

    w = w - learning_rate * dw
    b = b - learning_rate * db

In [38]:
X_test_value = X_test["Square_Footage"].values
test_predictions = make_prediction(X_test_value, w, b)


print("Slope m :", w)
print("Intercept c:", b)

Slope m : 216.6439999052017
Intercept c: 0.24349352609572059


---
## Question 4 (10 Marks)

Build a **ColumnTransformer** that applies:
- `StandardScaler` on numerical columns: `Square_Footage`, `Num_Bedrooms`, `Num_Bathrooms`
- `OneHotEncoder` on categorical column: `Neighborhood_Quality`



In [43]:
#Question 4
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.pipeline import Pipeline

In [46]:
# Convert to categorical
df["Neighborhood_Quality"] = df["Neighborhood_Quality"].astype(str)



num_cols = ["Square_Footage", "Num_Bedrooms", "Num_Bathrooms"]
cat_cols = ["Neighborhood_Quality"]

In [52]:
preprocessor = ColumnTransformer(
    transformers=[
        ("num", StandardScaler(), num_cols),
        ("cat", OneHotEncoder(), cat_cols)
    ],
    remainder="passthrough"
)
preprocessor



ColumnTransformer(remainder='passthrough',
                  transformers=[('num', StandardScaler(),
                                 ['Square_Footage', 'Num_Bedrooms',
                                  'Num_Bathrooms']),
                                ('cat', OneHotEncoder(),
                                 ['Neighborhood_Quality'])])

In [50]:
X_new = preprocessor.fit_transform(df)

print(X_new.shape)

(1000, 13)


## Question 5 (20 Marks)

Build a complete **Pipeline** using Scikit-Learn that includes:
- The `ColumnTransformer`
- `SGDRegressor` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples

In [53]:
# Question 5
from sklearn.linear_model import LinearRegression,SGDRegressor
from sklearn.metrics import mean_squared_error,r2_score,root_mean_squared_error,mean_absolute_error

In [65]:
LR_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", SGDRegressor(random_state=42))
])

LR_pipe

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['Square_Footage',
                                                   'Num_Bedrooms',
                                                   'Num_Bathrooms']),
                                                 ('cat', OneHotEncoder(),
                                                  ['Neighborhood_Quality'])])),
                ('model', SGDRegressor(random_state=42))])

In [66]:
LR_pipe.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['Square_Footage',
                                                   'Num_Bedrooms',
                                                   'Num_Bathrooms']),
                                                 ('cat', OneHotEncoder(),
                                                  ['Neighborhood_Quality'])])),
                ('model', SGDRegressor(random_state=42))])

In [67]:
y_pred = LR_pipe.predict(X_test)
y_pred[:10]



array([-1.11175082e+15, -1.09620945e+15, -1.08200917e+15, -1.08757736e+15,
       -1.09971879e+15, -1.08317209e+15, -1.10311909e+15, -1.09866777e+15,
       -1.09700916e+15, -1.10387723e+15])

In [72]:

print(f"R2: {round(r2_score(y_test, y_pred), 4)}")
print(f"RMSE: {round(root_mean_squared_error(y_test, y_pred), 4)}")


R2: 0.9984
RMSE: 10250.3104


---
## Question 6 (20 Marks)

Implement **Multiple Linear Regression** using **Scikit-Learn**:
- The `ColumnTransformer`
- `LinearRegression` as the final estimator
- Train the pipeline and evaluate using RMSE and R² score
- Print predicted vs actual values for the first 10 test samples

In [68]:
# Question 6
SGD_pipe = Pipeline([
    ("preprocessor", preprocessor),
    ("model", LinearRegression())
])

SGD_pipe

Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['Square_Footage',
                                                   'Num_Bedrooms',
                                                   'Num_Bathrooms']),
                                                 ('cat', OneHotEncoder(),
                                                  ['Neighborhood_Quality'])])),
                ('model', LinearRegression())])

In [69]:
SGD_pipe.fit(X_train, y_train)

/usr/local/lib/python3.12/dist-packages/sklearn/compose/_column_transformer.py:1667: FutureWarning: 
The format of the columns of the 'remainder' transformer in ColumnTransformer.transformers_ will change in version 1.7 to match the format of the other transformers.
At the moment the remainder columns are stored as indices (of type int). With the same ColumnTransformer configuration, in the future they will be stored as column names (of type str).
To use the new behavior now and suppress this warning, use ColumnTransformer(force_int_remainder_cols=False).

  warnings.warn(


Pipeline(steps=[('preprocessor',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('num', StandardScaler(),
                                                  ['Square_Footage',
                                                   'Num_Bedrooms',
                                                   'Num_Bathrooms']),
                                                 ('cat', OneHotEncoder(),
                                                  ['Neighborhood_Quality'])])),
                ('model', LinearRegression())])

In [70]:
y_pred = SGD_pipe.predict(X_test)
y_pred[:10]

array([ 869874.43985063,  493008.02345424,  943543.84812462,
       1032332.23749615,  776180.75134111,  733011.7995851 ,
        992374.7738462 ,  885753.69473635,  796210.75661298,
        932031.02566725])

In [71]:
print(f"R2: {round(r2_score(y_test, y_pred), 4)}")
print(f"RMSE: {round(root_mean_squared_error(y_test, y_pred), 4)}")


R2: 0.9984
RMSE: 10250.3104


---
## Question 7 (10 Marks) (You have to explore the topic and use the equation via Numpy)
### Dont use LLMs , You can use Documentation

Implement **Multiple Linear Regression** using **only NumPy**:
- Pick random 100 datas from the dataset
- Use the Normal Equation: `θ = (XᵀX)⁻¹ Xᵀy`
- Use `Square_Footage`, `Num_Bedrooms`, and `Num_Bathrooms` as features
- Print the learned coefficients (θ values)

In [74]:
# Question 7
sample_df = df.sample(100, random_state=42)

sample_df.head()

,Square_Footage,Num_Bedrooms,Num_Bathrooms,Year_Built,Lot_Size,Garage_Size,Neighborhood_Quality,House_Price
521,4012.0,3.0,1.0,2016.0,2.098092,1.0,5.0,9.010005e+05
737,2310.0,3.0,1.0,1988.0,1.369622,1.0,4.0,4.945375e+05
740,4708.0,1.0,3.0,1962.0,1.792970,1.0,8.0,9.494042e+05
660,4932.0,2.0,1.0,1972.0,4.479598,1.0,2.0,1.040389e+06
411,3646.0,1.0,1.0,1994.0,3.980987,0.0,9.0,7.940100e+05


In [78]:
X = sample_df[["Square_Footage", "Num_Bedrooms", "Num_Bathrooms"]].values
y = sample_df["House_Price"].values


def normal_equation(X, y):

    # Calculates theta using the Normal Equation
    # θ = (XᵀX)⁻¹Xᵀy


    m = X.shape[0]
    X = np.c_[np.ones((m, 1)), X]

    XT = X.T

    theta = np.linalg.inv(XT @ X) @ XT @ y
    return theta


In [77]:
theta = normal_equation(X, y)

print("Theta Values:")
print(theta)

print("Intercept :", theta[0])
print("Square_Footage :", theta[1])
print("Num_Bedrooms :", theta[2])
print("Num_Bathrooms :", theta[3])

Theta Values:
[20888.25945218   202.30820792  8303.2810412   3020.32852173]
Intercept : 20888.259452177677
Square_Footage : 202.30820792026844
Num_Bedrooms : 8303.281041200218
Num_Bathrooms : 3020.3285217270677
